# AzureOpenAI를 활용한 Chat Completions

이 노트북에서는 **Microsoft Foundry** SDK를 사용하여 **Chat Completions**을 수행하는 방법을 보여드립니다. **`azure-ai-projects`** 와 **`azure-ai-inference`** 패키지를 조합하여 다음을 수행합니다:

1. `ChatCompletionsClient`를 **초기화**합니다.   
2. 시스템 컨텍스트를 추가하기 위해 **프롬프트 템플릿을 사용**합니다.  
3. 건강 & 피트니스 주제의 사용자 프롬프트를 **전송**합니다.  

## 🏋️ 건강-피트니스 관련 안내  
> **이 예제는 데모 목적일 뿐이며 실제 의학적 조언을 제공하지 않습니다.**  
> 건강이나 의학 관련 질문은 반드시 전문가와 상담하세요.

### 사전 준비 사항  
이 노트북을 시작하기 전에 루트 [README.md](./../README.md)에 명시된 모든 사전 조건을 완료했는지 확인하세요.

그럼 시작해볼까요? 🎉


## 1. 초기 설정  
환경 변수를 로드하고, `ChatCompletionsClient`를 생성합니다.
또한 시스템 메시지를 구성하는 방식을 보여주기 위해 **프롬프트 템플릿**도 정의할 예정입니다.


In [10]:
import os
from dotenv import load_dotenv
from pathlib import Path

from azure.core.credentials import AzureKeyCredential

from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import UserMessage, SystemMessage

# Load environment variables
def _clean_env(value: str | None) -> str | None:
    if value is None:
        return None
    return value.strip().strip("\"").strip("'")

dotenv_candidates = [
    Path.cwd() / '.env',
    Path.cwd().parent / '.env',
    Path('.env'),
    Path('..') / '.env',
]
dotenv_path = next((p for p in dotenv_candidates if p.exists()), None)
if dotenv_path:
    load_dotenv(dotenv_path, override=False)
    print(f"✅ Loaded .env from: {dotenv_path}")
else:
    print("⚠️ .env file not found. Tried current/parent directories.")

# Retrieve from environment
endpoint = _clean_env(os.environ.get("ENDPOINT"))
api_key = _clean_env(os.environ.get("API_KEY"))
chat_model = _clean_env(os.environ.get("MODEL_NAME"))

chat_client = None
missing_keys = [
    name
    for name, value in {
        "ENDPOINT": endpoint,
        "API_KEY": api_key,
        "MODEL_NAME": chat_model,
    }.items()
    if not value
]
if missing_keys:
    print(f"❌ Missing required env keys: {', '.join(missing_keys)}")
else:
    try:
        chat_client = ChatCompletionsClient(
            endpoint=endpoint,
            credential=AzureKeyCredential(api_key),
            model=chat_model
        )
        print("✅ Successfully created AzureOpenAI")
    except Exception as e:
        print("❌ Error initializing client:", e)

✅ Loaded .env from: c:\VSCode\AzureMSFoundryWorkshop-Code\.env
✅ Successfully created AzureOpenAI


### 프롬프트 템플릿  
친절하고 안내 문구를 제공하는 피트니스 어시스턴트 역할의 **시스템** 메시지를 간단히 정의해보겠습니다.

```txt
시스템 프롬프트 (템플릿):
당신은 FitChat GPT, 도움이 되는 피트니스 어시스턴트입니다.  
항상 사용자에게 상기시키세요: 저는 의료 전문가는 아닙니다.  
친절하게 응답하고, 일반적인 조언을 제공합니다.  
...
```

그런 다음 사용자 입력을 **user** 메시지로 전달할 것입니다.


In [7]:
# We'll define a function that runs chat completions with a system prompt & user prompt
def chat_with_fitness_assistant(user_input: str):
    """Use chat completions to get a response from our LLM, with system instructions."""
    if chat_client is None:
        raise RuntimeError("chat_client is not initialized. Check ENDPOINT/API_KEY/MODEL_NAME in .env")
    # Our system message template
    system_text = (
        "You are FitChat GPT, a friendly fitness assistant.\n"
        "Always remind users: I'm not a medical professional.\n"
        "Answer with empathy and disclaimers."
    )

    response = chat_client.complete(
        messages=[
            SystemMessage(content=system_text),
            UserMessage(content=user_input)
        ],
    )

    return response.choices[0].message.content

print("Defined a helper function to do chat completions.")

Defined a helper function to do chat completions.


## 2. Chat Completions 실행해보기 
건강이나 피트니스에 관한 사용자의 질문으로 함수를 호출하고, 결과를 확인해보겠습니다.  
질문을 자유롭게 수정하거나 여러 번 실행해보셔도 좋습니다!


In [8]:
user_question = "How can I start a beginner workout routine at home?"
reply = chat_with_fitness_assistant(user_question)
print("🗣️ User:", user_question)
print("🤖 Assistant:", reply)

ServiceRequestError: HTTPSConnection(host='wonsungso-0304.services.ai.azure.com', port=443): Failed to resolve 'wonsungso-0304.services.ai.azure.com' ([Errno 11001] getaddrinfo failed)

## 3. 채우기 형식의 프롬프트 템플릿 
시스템 메시지에 플레이스홀더를 추가하여 좀 더 확장된 형태로 구성할 수 있습니다.  
예를 들어, **userName**이나 **goal**과 같은 변수가 있다고 가정해보세요.  
간단한 예제를 통해 보여드리겠습니다.

In [9]:
def chat_with_template(user_input: str, user_name: str, goal: str):
    if chat_client is None:
        raise RuntimeError("chat_client is not initialized. Check ENDPOINT/API_KEY/MODEL_NAME in .env")
    # Construct a system template with placeholders
    system_template = (
        "You are FitChat GPT, an AI personal trainer for {name}.\n"
        "Your user wants to achieve: {goal}.\n"
        "Remind them you're not a medical professional. Offer friendly advice."
    )

    # Fill in placeholders
    system_prompt = system_template.format(name=user_name, goal=goal)

    response = chat_client.complete(
        messages=[
            SystemMessage(content=system_prompt),
            UserMessage(content=user_input)
        ],
    )

    return response.choices[0].message.content

# Let's try it out
templated_user_input = "What kind of home exercise do you recommend for a busy schedule?"
assistant_reply = chat_with_template(
    templated_user_input,
    user_name="Jordan",
    goal="increase muscle tone and endurance"
)
print("🗣️ User:", templated_user_input)
print("🤖 Assistant:", assistant_reply)

ServiceRequestError: HTTPSConnection(host='wonsungso-0304.services.ai.azure.com', port=443): Failed to resolve 'wonsungso-0304.services.ai.azure.com' ([Errno 11001] getaddrinfo failed)